In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t022/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
```python
"The alert is titled 'inventory接口流量下跌告警' (inventory API traffic drop alert). It occurred between 2026-04-13 06:21:20 and 2026-04-13 06:28:54 (UTC+8). The entity that triggered the alert is an APM operation named 'inventory::/api/v1/inventory/{productId}' with the ID '162799adab6799326676266a2c96cf3e'."
```

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-04-13T06:21:20+08:00', 'end': '2026-04-13T06:28:54+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t022/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all unique KPIs (metric names) and entity sets
unique_kpis = metrics_data['metric'].unique()
unique_entity_sets = metrics_data['entity_set'].unique()

unique_kpis, unique_entity_sets

```
Out[2]:
```
```python
"The metrics data contains the following unique KPIs: node-level metrics (e.g., 'node_cpu_usage_rate', 'node_memory_usage_rate'), deployment-level metrics (e.g., 'deployment_available_replicas', 'deployment_cpu_usage_total'), cluster-level metrics (e.g., 'cluster_node_count', 'cluster_failed_node_count'), namespace-level metrics (e.g., 'namespace_number_of_pods'), APM operation metrics (e.g., 'error_count', 'request_count', 'latency'), JVM metrics (e.g., 'arms_jvm_gc_total', 'arms_jvm_mem_used_bytes'), thread metrics (e.g., 'arms_thread_pool_active_thread_count'), and service-level metrics (e.g., 'error_rate', 'slow').

The available entity sets include: 'k8s.node', 'k8s.deployment', 'k8s.cluster', 'k8s.namespace', 'apm.operation', 'apm.instance', 'apm.metric.jvm', 'apm.metric.thread', and 'apm.service.legacy'."
```

The original code execution output of IPython Kernel is also provided below for reference:

(array(['node_ready_status', 'node_memory_usage_rate',
       'node_not_ready_status', 'node_pod_running_count',
       'node_cpu_usage_rate', 'node_disk_usage_rate',
       'deployment_available_replicas', 'deployment_ready_replicas',
       'deployment_desired_replicas', 'deployment_availability_rate',
       'deployment_memory_usage_total',
       'deployment_cpu_usage_vs_requests',
       'deployment_cpu_usage_vs_limits', 'deployment_cpu_usage_total',
       'cluster_failed_node_count', 'cluster_node_count',
       'cluster_number_of_pod', 'cluster_number_of_not_ready_pods',
       'apiserver_non_2xx_read_request_qps',
       'apiserver_concurrent_connections',
       'apiserver_non_2xx_write_request_qps',
       'apiserver_requests_per_second',
       'namespace_number_of_not_ready_pods', 'namespace_number_of_pods',
       'error_count', 'slow_count', 'request_count',
       'avg_request_latency_seconds', 'arms_jvm_gc_total',
       'arms_jvm_gc_seconds_total', 'arms_jvm_gc_delta',
       'arms_jvm_gc_seconds_delta', 'arms_jvm_mem_used_bytes',
       'arms_jvm_mem_init_bytes', 'arms_jvm_mem_committed_bytes',
       'arms_jvm_mem_max_bytes', 'arms_jvm_threads_count',
       'arms_thread_pool_active_thread_count',
       'arms_thread_pool_current_thread_count',
       'arms_thread_pool_max_thread_count',
       'arms_thread_pool_max_pool_size',
       'arms_thread_pool_core_pool_size',
       'arms_thread_pool_scheduled_task_count',
       'arms_thread_pool_completed_task_count',
       'arms_thread_pool_rejected_task_count',
       'arms_thread_pool_queue_size', 'arms_grouped_thread_count',
       'arms_grouped_thread_cpu_time',
       'arms_grouped_thread_runnable_count',
       'arms_grouped_thread_waiting_count',
       'arms_grouped_thread_timed_waiting_count', 'cpu', 'latency', 'mem',
       'workload', 'slow', 'error', 'error_rate'], dtype=object), array(['k8s.node', 'k8s.deployment', 'k8s.cluster', 'k8s.namespace',
       'apm.operation', 'apm.instance', 'apm.metric.jvm',
       'apm.metric.thread', 'apm.service.legacy'], dtype=object))```
```

In [ ]:
# In[3]:

# Filter the metrics data for 'apm.operation' entity set and the specific entity name
filtered_metrics = metrics_data[
    (metrics_data['entity_set'] == 'apm.operation') &
    (metrics_data['entity_name'] == 'inventory::/api/v1/inventory/{productId}')
]

# Convert 'value' column to numeric for calculations
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Filter for the relevant KPIs
relevant_kpis = filtered_metrics[filtered_metrics['metric'].isin(['error_count', 'request_count', 'latency'])]

# Calculate global thresholds (P95) for each KPI
global_thresholds = relevant_kpis.groupby('metric')['value'].quantile(0.95)

global_thresholds

```
Out[3]:
```
```python
"Among the KPIs for the operation 'inventory::/api/v1/inventory/{productId}', only 'request_count' has sufficient data to calculate a global threshold. The P95 value for 'request_count' is 3976.45. No data was available for 'error_count' or 'latency' to compute their thresholds."
```

The original code execution output of IPython Kernel is also provided below for reference:

metric
request_count    3976.45
Name: value, dtype: float64```
```

In [ ]:
# In[4]:

import pytz
from datetime import datetime

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat('2026-04-13T06:21:20+08:00').astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat('2026-04-13T06:28:54+08:00').astimezone(pytz.timezone('Asia/Shanghai'))

# Convert the 'time' column to numeric and filter for the alert time window
filtered_metrics['time'] = pd.to_numeric(filtered_metrics['time'], errors='coerce')
alert_window_data = filtered_metrics[
    (filtered_metrics['metric'] == 'request_count') &
    (filtered_metrics['time'] >= alert_start.timestamp() * 1_000_000) &
    (filtered_metrics['time'] <= alert_end.timestamp() * 1_000_000)
]

# Identify anomalies where 'request_count' falls below the global P95 threshold
anomalies = alert_window_data[alert_window_data['value'] < 3976.45]

anomalies

```
Out[4]:
```
```python
"During the alert time window (2026-04-13 06:21:20 to 2026-04-13 06:28:54 UTC+8), several anomalies were detected in the 'request_count' metric for the operation 'inventory::/api/v1/inventory/{productId}'. The values fell below the global P95 threshold of 3976.45 multiple times, with the lowest value being 869.0. This indicates a significant drop in traffic for the inventory API during the alert period."
```

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                               entity_name         metric   value                            metric_set_id service
62325  1776032480000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3369.0  apm@metric_set@apm.metric.apm.operation    None
62326  1776032510000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3913.0  apm@metric_set@apm.metric.apm.operation    None
62328  1776032570000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3486.0  apm@metric_set@apm.metric.apm.operation    None
62329  1776032600000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3105.0  apm@metric_set@apm.metric.apm.operation    None
62330  1776032630000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3504.0  apm@metric_set@apm.metric.apm.operation    None
62331  1776032660000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3263.0  apm@metric_set@apm.metric.apm.operation    None
62332  1776032690000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3042.0  apm@metric_set@apm.metric.apm.operation    None
62333  1776032720000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3475.0  apm@metric_set@apm.metric.apm.operation    None
62334  1776032750000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  2770.0  apm@metric_set@apm.metric.apm.operation    None
62335  1776032780000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count   869.0  apm@metric_set@apm.metric.apm.operation    None
62336  1776032810000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  1388.0  apm@metric_set@apm.metric.apm.operation    None
62337  1776032840000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3013.0  apm@metric_set@apm.metric.apm.operation    None
62338  1776032870000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3092.0  apm@metric_set@apm.metric.apm.operation    None
62339  1776032900000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3399.0  apm@metric_set@apm.metric.apm.operation    None
62340  1776032930000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  3330.0  apm@metric_set@apm.metric.apm.operation    None```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t022/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert 'startTime' and 'endTime' to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces for the operation 'inventory::/api/v1/inventory/{productId}' during the alert time window
alert_start_ns = int(alert_start.timestamp() * 1_000_000_000)  # Convert to nanoseconds
alert_end_ns = int(alert_end.timestamp() * 1_000_000_000)      # Convert to nanoseconds

filtered_traces = traces_data[
    (traces_data['spanName'] == 'inventory::/api/v1/inventory/{productId}') &
    (traces_data['startTime'] >= alert_start_ns) &
    (traces_data['startTime'] <= alert_end_ns)
]

# Extract relevant columns: trace IDs, span names, durations, and status codes
relevant_trace_data = filtered_traces[['traceId', 'spanName', 'duration', 'statusCode']]

relevant_trace_data

```
Out[5]:
```
```python
"No traces were found for the operation 'inventory::/api/v1/inventory/{productId}' during the alert time window (2026-04-13 06:21:20 to 2026-04-13 06:28:54 UTC+8). This indicates that no trace data is available to analyze errors or slow traces for this operation during the specified period."
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanName, duration, statusCode]
Index: []```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t022/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert '_time_' column to datetime for filtering
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs for the operation 'inventory::/api/v1/inventory/{productId}' during the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start) &
    (logs_data['_time_'] <= alert_end) &
    (logs_data['content'].str.contains('inventory::/api/v1/inventory/{productId}', na=False))
]

# Search for error messages or anomalies in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
```python
"No logs containing error messages or anomalies were found for the operation 'inventory::/api/v1/inventory/{productId}' during the alert time window (2026-04-13 06:21:20 to 2026-04-13 06:28:54 UTC+8). This indicates that no relevant log data is available to analyze issues for this operation during the specified period."
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```